# Bronze ingestion

Set the widgets to Unity Catalog volume paths for the validated input and Delta outputs. The local runner injects equivalent filesystem paths.


In [ ]:
if "batch_root" not in globals():
    dbutils.widgets.text("batch_root", "/Volumes/workspace/default/finops/input")
    dbutils.widgets.text("lake_root", "/Volumes/workspace/default/finops/lake")
    batch_root, lake_root = [dbutils.widgets.get(name).rstrip("/") for name in ("batch_root", "lake_root")]

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
for kind in ("events", "prices", "scores"):
    path = f"{batch_root}/{kind}.jsonl"
    payloads = spark.read.text(path).filter(F.length(F.trim("value")) > 0).select(
        F.col("value").alias("payload"), F.lit(path).alias("source_file"),
        F.current_timestamp().alias("ingested_at"),
    )
    bronze_path = f"{lake_root}/bronze/{kind}"
    payloads.write.format("delta").mode("overwrite").save(bronze_path)
    print(kind, spark.read.format("delta").load(bronze_path).count())
